# EDA Dataset House Prices (Ames Housing) — Panduan Lengkap

**Sub-CPMK:** **P4** — EDA univariat/bivariat; visualisasi dengan Matplotlib & Seaborn.

Notebook ini adalah **tutorial mandiri** (melengkapi `minggu_05.ipynb` dan menjembatani **regresi** di `minggu_07.ipynb`): eksplorasi tabel, missing values, distribusi target kontinu `SalePrice`, hubungan prediktor–harga, dan hipotesis untuk model regresi.

**Konteks data:** [Kaggle — House Prices: Advanced Regression Techniques](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques) / **Ames Housing** (Dean De Cock). **1460** penjualan rumah di Ames, Iowa (~2006–2010), **~80** fitur per observasi.

**Target:** `SalePrice` — harga jual (USD), variabel **kontinu** (regresi, bukan klasifikasi).

**Sumber data:** `sklearn.datasets.fetch_openml(name="house_prices")` — selaras `minggu_02.ipynb` (OpenML). **Unduhan pertama membutuhkan internet.**

**Pertanyaan analitik:**
- Apakah `SalePrice` simetris atau skew? Apakah transformasi log membantu?
- Fitur numerik mana yang paling berkorelasi dengan harga?
- Seberapa besar perbedaan harga antar `Neighborhood` dan `OverallQual`?
- Pola missing pada kolom basemen/kolam — acak atau terstruktur?

**Referensi:**
- [LearnPython — Ames data analysis](https://learnpython.com/blog/python-data-analysis-example/)
- [Alberto Bas — Ames housing EDA](https://www.albertobas.com/blog/ames-housing-prices-eda)
- [Nitin Gupta — Ames Part 1 EDA](https://nitingupta2.github.io/casestudies/ames-housing-part1-eda/)
- [DEV — EDA with Seaborn](https://dev.to/nexttech/how-to-perform-exploratory-data-analysis-with-seaborn-29eo)
- [Kaggle — data description](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques/data)
- Modul PDF: `modul-05.tex` (Minggu 5); regresi: `minggu_07.ipynb` (RMSE/MAE)


## 0. Kerangka CRISP-DM dan kamus fitur inti

Fase **Data Understanding** (CRISP-DM): pahami variabel sebelum memplot. EDA **iteratif** — temuan mengarahkan imputasi (`minggu_03`), encoding/skala (`minggu_04`), dan regresi (`minggu_07`).

Dataset penuh ~**79** prediktor + `SalePrice`. Definisi resmi setiap kolom: [Kaggle data_description.pdf](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques/data).

| Fitur | Arti singkat |
|-------|----------------|
| `SalePrice` | Harga jual rumah (USD) — **target** |
| `OverallQual` | Kualitas keseluruhan material/akab (1–10) |
| `GrLivArea` | Luas hunian di atas tanah (kaki persegi) |
| `TotalBsmtSF` | Luas total basement (kaki persegi) |
| `LotArea` | Luas lot (kaki persegi) |
| `GarageCars` | Kapasitas garasi (mobil) |
| `FullBath` | Jumlah kamar mandi penuh di atas grade |
| `YearBuilt` | Tahun bangunan |
| `YearRemodAdd` | Tahun renovasi/d tambahan |
| `Neighborhood` | Nama lingkungan fisik di Ames |
| `MSZoning` | Klasifikasi zonasi |
| `HouseStyle` | Gaya arsitektur rumah |

Notebook ini fokus pada fitur di atas + eksplorasi missing global; analisis mendalam 79 kolom dilakukan bertahap di proyek Kaggle.


## 1. Persiapan lingkungan

Import pustaka standar praktikum; muat data dari OpenML.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_openml

RANDOM_STATE = 42
TARGET = "SalePrice"
sns.set_theme(style="whitegrid", context="notebook")
%matplotlib inline

**Memuat Ames Housing dari OpenML.** `parser="auto"` membantu kompatibilitas tipe kolom; `.copy()` untuk salinan independen.


In [ ]:
print("Mengunduh/memuat house_prices dari OpenML (internet pada unduhan pertama)...")
ames = fetch_openml(name="house_prices", version=1, as_frame=True, parser="auto")
df = ames.frame.copy()
print("Bentuk data:", df.shape)
df.head()

**Cuplikan acak, info, dan audit awal.** Kolom `Id` bukan prediktor; kita pisahkan tipe numerik vs kategorikal untuk EDA terarah.


In [ ]:
print("Cuplikan acak 15 rumah:")
display(df.sample(15, random_state=RANDOM_STATE))

print("\nDuplikat baris penuh:", df.duplicated().sum())
df.info()

**Pemisahan kolom untuk analisis.** Fitur prediktor = semua kolom kecuali `Id` dan target.


In [ ]:
drop_id = df.drop(columns=["Id"], errors="ignore")
feature_cols = [c for c in drop_id.columns if c != TARGET]

numeric_cols = drop_id[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = drop_id[feature_cols].select_dtypes(include=["object", "category"]).columns.tolist()

print(f"Prediktor numerik: {len(numeric_cols)}")
print(f"Prediktor kategorikal: {len(categorical_cols)}")
print("\nContoh numerik:", numeric_cols[:8], "...")
print("Contoh kategorikal:", categorical_cols[:8], "...")

## 2. EDA tabular (Part I)

Ringkasan statistik, missing values, dan korelasi dengan `SalePrice` sebelum visual.


**Statistik deskriptif** — numerik dan kategorikal (modus, frekuensi).


In [ ]:
display(df[TARGET].describe().to_frame().T)
display(df[numeric_cols].describe().T.head(12))

display(df[categorical_cols].describe(include="object").T.head(10))

**Mean vs median (skew).** Jika mean >> median, distribusi cenderung skew kanan (outlier atas).


In [ ]:
skew_check = pd.DataFrame({
    "mean": df[[TARGET, "GrLivArea", "LotArea", "TotalBsmtSF"]].mean(),
    "median": df[[TARGET, "GrLivArea", "LotArea", "TotalBsmtSF"]].median(),
})
skew_check["mean_minus_median"] = skew_check["mean"] - skew_check["median"]
display(skew_check.round(2))

**Missing values** — Ames banyak kolom dengan NA yang berarti "tidak ada fitur" (bukan kesalahan entri).


In [ ]:
missing_count = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing_count / len(df) * 100).round(2)
missing_tbl = pd.DataFrame({"jumlah_na": missing_count, "persen": missing_pct})
missing_tbl[missing_tbl["jumlah_na"] > 0].head(20)

**Visual missing** — kolom dengan NA terbanyak (PoolQC, Alley, Fence, …).


In [ ]:
miss = missing_tbl[missing_tbl["jumlah_na"] > 0].sort_values("jumlah_na", ascending=True).tail(15)
plt.figure(figsize=(8, 5))
sns.barplot(x=miss["jumlah_na"], y=miss.index, hue=miss.index, palette="Oranges_d", legend=False)
plt.title("15 kolom dengan missing terbanyak")
plt.xlabel("Jumlah NA")
plt.ylabel("Kolom")
plt.tight_layout()
plt.show()

**Heatmap pola missing** (setiap baris = satu rumah).


In [ ]:
cols_miss = missing_tbl[missing_tbl["jumlah_na"] > 0].head(25).index.tolist()
plt.figure(figsize=(12, 6))
sns.heatmap(df[cols_miss].isna(), cbar=False, yticklabels=False, cmap="magma")
plt.title("Pola NA pada 25 kolom teratas (baris = observasi)")
plt.xlabel("Kolom")
plt.tight_layout()
plt.show()

**Interpretasi (missing):** NA pada `PoolQC`, `Alley`, `Fence` sering berarti fasilitas tidak ada — perlu strategi imputasi khusus (kategori "None"), bukan median numerik. Lihat `minggu_03.ipynb`.


**Korelasi numerik dengan SalePrice** — urutkan menurut |korelasi|.


In [ ]:
corr_sale = df[numeric_cols + [TARGET]].corr(numeric_only=True)[TARGET].drop(TARGET)
corr_sale = corr_sale.sort_values(key=abs, ascending=False)
print("Top 15 korelasi dengan SalePrice:")
display(corr_sale.head(15).round(3).to_frame("corr"))

**Interpretasi (tabel):** `OverallQual`, `GrLivArea`, `GarageCars`, `TotalBsmtSF` berada di puncak korelasi; banyak kolom lemah — feature selection/engineering penting sebelum regresi.


## 3. EDA target — `SalePrice`

Distribusi harga jual: skew, kurtosis, dan demo transformasi log (Alberto Bas).


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(df[TARGET], kde=True, ax=ax[0])
ax[0].set_title("Distribusi SalePrice")
ax[0].set_xlabel("SalePrice (USD)")
ax[0].set_ylabel("Frekuensi")

log_price = np.log1p(df[TARGET])
sns.histplot(log_price, kde=True, ax=ax[1], color="seagreen")
ax[1].set_title("Distribusi log1p(SalePrice)")
ax[1].set_xlabel("log(1 + SalePrice)")
plt.tight_layout()
plt.show()

print("Skew SalePrice:", df[TARGET].skew().round(3))
print("Kurtosis SalePrice:", df[TARGET].kurt().round(3))
print("Skew log1p(SalePrice):", log_price.skew().round(3))

**Interpretasi (target):** `SalePrice` skew kanan (mean > median); `log1p` mendekatkan bentuk ke simetris — regresi linear pada log-harga sering lebih stabil (`minggu_07`).


## 4. EDA univariat — prediktor kunci

Distribusi fitur yang paling sering dipakai dalam analisis Ames.


In [ ]:
key_num = ["OverallQual", "GrLivArea", "LotArea", "TotalBsmtSF", "YearBuilt", "GarageCars"]
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for ax, col in zip(axes.ravel(), key_num):
    sns.histplot(df[col].dropna(), kde=True, ax=ax)
    ax.set_title(col)
    ax.set_xlabel(col)
plt.suptitle("Histogram prediktor numerik/ordinal utama", y=1.02)
plt.tight_layout()
plt.show()

**Kategorikal:** frekuensi `Neighborhood` (top 12), `MSZoning`, `CentralAir`.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
top_nb = df["Neighborhood"].value_counts().head(12).index
sns.countplot(data=df[df["Neighborhood"].isin(top_nb)], x="Neighborhood", order=top_nb, ax=axes[0])
axes[0].set_title("Neighborhood (top 12)")
axes[0].tick_params(axis="x", rotation=45)

sns.countplot(data=df, x="MSZoning", ax=axes[1])
axes[1].set_title("MSZoning")
axes[1].tick_params(axis="x", rotation=45)

sns.countplot(data=df, x="CentralAir", ax=axes[2])
axes[2].set_title("CentralAir")
plt.tight_layout()
plt.show()

## 5. EDA bivariat — prediktor vs `SalePrice`

Setiap plot: judul, label sumbu, interpretasi singkat.


**Luas hunian vs harga** — perhatikan outlier (GrLivArea > 4000 sf).


In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="GrLivArea", y=TARGET, alpha=0.5, hue="OverallQual", palette="viridis", legend=False)
plt.title("GrLivArea vs SalePrice (warna: OverallQual)")
plt.xlabel("GrLivArea (sf)")
plt.ylabel("SalePrice (USD)")
plt.tight_layout()
plt.show()

**Interpretasi:** Hubungan hampir linear positif; beberapa titik ekstrem (luas sangat besar) menarik harga ke atas — kandidat outlier untuk investigasi atau robust regression.


**Kualitas keseluruhan vs harga.**


In [ ]:
plt.figure(figsize=(7, 4))
sns.scatterplot(data=df, x="OverallQual", y=TARGET, alpha=0.55)
plt.title("OverallQual vs SalePrice")
plt.xlabel("OverallQual (1-10)")
plt.ylabel("SalePrice (USD)")
plt.tight_layout()
plt.show()

**Interpretasi:** Peningkatan per level `OverallQual` jelas mengangkat harga — prediktor kuat untuk model baseline.


**Harga per Neighborhood** (top 12 frekuensi).


In [ ]:
top_nb = df["Neighborhood"].value_counts().head(12).index
plt.figure(figsize=(10, 5))
sns.boxplot(data=df[df["Neighborhood"].isin(top_nb)], x="Neighborhood", y=TARGET, order=top_nb)
plt.title("SalePrice per Neighborhood (top 12)")
plt.xlabel("Neighborhood")
plt.ylabel("SalePrice (USD)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

**Interpretasi:** Median harga sangat berbeda antar lingkungan (mis. Northridge vs MeadowV) — encoding `Neighborhood` atau target encoding relevan.


**Zoning, gaya rumah, dan waktu penjualan.**


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
sns.boxplot(data=df, x="MSZoning", y=TARGET, ax=axes[0])
axes[0].set_title("SalePrice vs MSZoning")
axes[0].tick_params(axis="x", rotation=45)

sns.boxplot(data=df, x="HouseStyle", y=TARGET, ax=axes[1])
axes[1].set_title("SalePrice vs HouseStyle")
axes[1].tick_params(axis="x", rotation=45)

sns.boxplot(data=df, x="YrSold", y=TARGET, ax=axes[2])
axes[2].set_title("SalePrice vs YrSold")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
sns.boxplot(data=df, x="MoSold", y=TARGET)
plt.title("SalePrice vs bulan penjualan (MoSold)")
plt.xlabel("MoSold")
plt.ylabel("SalePrice (USD)")
plt.tight_layout()
plt.show()

**Interpretasi:** Zoning residensial vs komersial memisahkan rentang harga; variasi tahun/bulan penjualan lebih kecil daripada lokasi/kualitas — efek musiman lemah dibanding `Neighborhood`.


## 6. EDA multivariat

Korelasi subset dan pairplot fitur terkuat.


In [ ]:
top_corr_cols = corr_sale.head(14).index.tolist() + [TARGET]
plt.figure(figsize=(10, 8))
sns.heatmap(df[top_corr_cols].corr(), annot=True, fmt=".2f", cmap="vlag", center=0)
plt.title("Heatmap korelasi — 14 prediktor teratas + SalePrice")
plt.tight_layout()
plt.show()

**Interpretasi:** `GarageCars` vs `GarageArea` dan `GrLivArea` vs `TotRmsAbvGrd` berpotensi multikolinear — hindari memasukkan keduanya tanpa sengaja ke model linear.


In [ ]:
pair_cols = [TARGET, "OverallQual", "GrLivArea", "TotalBsmtSF", "GarageArea"]
sns.pairplot(df[pair_cols].dropna(), corner=True, plot_kws={"alpha": 0.4, "s": 12})
plt.suptitle("Pairplot subset fitur vs SalePrice", y=1.02)
plt.show()

## 7. Fitur turunan (eksplorasi)

Harga per kaki persegi (ilustrasi) — bukan pipeline final.


In [ ]:
df_eda = df.copy()
df_eda["LivingAreaSf"] = df_eda["GrLivArea"] + df_eda["TotalBsmtSF"].fillna(0)
df_eda["PricePerSf"] = df_eda[TARGET] / df_eda["LivingAreaSf"].replace(0, np.nan)

nb_stats = (
    df_eda.groupby("Neighborhood", observed=True)
    .agg(median_price=(TARGET, "median"), median_ppsf=("PricePerSf", "median"), n=(TARGET, "count"))
    .query("n >= 5")
    .sort_values("median_price", ascending=False)
)
print("Top 5 Neighborhood — median harga & $/sf:")
display(nb_stats.head(5).round(0))
print("\nBottom 5 Neighborhood:")
display(nb_stats.tail(5).round(0))

**Interpretasi:** `PricePerSf` membantu membandingkan lokasi dengan ukuran rumah berbeda; StoneBr vs MeadowV bisa berjarak besar pada metrik ini (Alberto Bas).


## 8. Ringkasan temuan, bias, dan hipotesis pemodelan

### Ringkasan temuan (EDA)

1. **Target skew:** `SalePrice` tidak normal; `log1p` mengurangi skew untuk regresi.
2. **Prediktor kuat:** `OverallQual`, `GrLivArea`, `GarageCars`, `TotalBsmtSF` — korelasi tertinggi dengan harga.
3. **Lokasi:** `Neighborhood` menjelaskan variansi besar (boxplot).
4. **Missing terstruktur:** banyak NA = "tidak ada fitur"; jangan imputasi median sembarangan.
5. **Outlier:** rumah dengan `GrLivArea` ekstrem; perhatikan saat OLS/RMSE.
6. **Multikolinearitas:** pasangan garasi & ruang — pilih satu atau gunakan regularisasi (`Ridge` di `minggu_07`).
7. **Fitur turunan:** `PricePerSf`, luas total hunian — kandidat engineering Kaggle.

### Bias dan limitasi

- Data **hanya Ames, Iowa**, periode terbatas; tidak general ke pasar global.
- Harga mencerminkan kondisi 2006–2010 (termasuk krisis subprime).
- Model regresi memprediksi pola historis, bukan kausalitas "renovasi pasti naikkan harga X%".

### Hipotesis pemodelan (regresi)

Model **baseline** `log1p(SalePrice) ~ OverallQual + GrLivArea + C(Neighborhood)` dengan encoding one-hot dan scaling fitur numerik diharapkan mengalahkan prediksi harga median global. Evaluasi dengan **RMSE/MAE** pada skala log atau dollar setelah `expm1` — lihat `minggu_07.ipynb`.

### Langkah berikutnya

1. `minggu_03.ipynb` — imputasi NA terstruktur  
2. `minggu_04.ipynb` — encoding & `StandardScaler`  
3. `minggu_07.ipynb` — `LinearRegression`, RMSE, MAE  

Jalankan **Kernel → Restart & Run All** setelah koneksi internet tersedia untuk unduhan OpenML pertama.
